In [1]:
import logging
import time
import os
import pickle


import numpy as np
import matplotlib.pyplot as plt

#import tensorflow_datasets as tfds
import tensorflow as tf

# Import tf_text to load the ops used by the tokenizer saved model
#import tensorflow_text  # pylint: disable=unused-import
import pandas as pd
import numpy as np
import re
import seaborn as sns
import matplotlib as plt


from sklearn.model_selection import train_test_split


from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Model,  Sequential
from tensorflow.keras.layers import LSTM, GRU, Bidirectional, Dropout, Input, TimeDistributed, Dense, Activation, RepeatVector, Embedding, Concatenate
import tensorflow.keras.layers as layers
from tensorflow.keras.layers import Attention
from tensorflow.keras.optimizers import Adam, Adagrad
from keras.losses import sparse_categorical_crossentropy
logging.getLogger('tensorflow').setLevel(logging.ERROR)  # suppress warnings
import random

In [2]:
with open("PichiaData_2Target.pkl", "rb") as fp:
    Data_AllOrg = pickle.load(fp)
    
AA_tr = Data_AllOrg['AA_tr']
Cds_tr = Data_AllOrg['Cds_tr']
AA_ts = Data_AllOrg['AA_ts']
Cds_ts = Data_AllOrg['Cds_ts']

Settings = pd.read_csv('./BO_forHyperParameter/Arch1/Round1.csv').iloc[:, 1:]
Setting_no = 2


    
Max_length = 1000
learning_rate = 0.001
batch_size = 150
epochs = 100
aa_vocab_size = 25
dna_vocab_size = 67


hidden_size_enc = int(Settings['Enc hidden size'][Setting_no])
hidden_size_enc_aa = int(Settings['Enc hidden size'][Setting_no])
embedding_size_enc = int(Settings['Enc Embedding size'][Setting_no])
embedding_size_dec = int(Settings['Dec Embedding size'][Setting_no])
Dense_layer_size = int(Settings['Dense Layer size'][Setting_no])
Dense_layer_size_aa = int(Settings['Dense Layer size aa'][Setting_no])

drop_rate = Settings['Drop rate'][Setting_no]
drop_rate_aa = Settings['Drop rate aa'][Setting_no]


#### Network parameters -  2 outputs

In [3]:
input_sequence = Input(shape=(Max_length,))
encod_emb = Embedding(input_dim= aa_vocab_size, output_dim = embedding_size_enc,trainable=True, mask_zero = True)
embedding = encod_emb(input_sequence)

encoder = Bidirectional(GRU(hidden_size_enc, return_sequences=True, return_state = True),
                        merge_mode="concat", weights=None)

encoder_sequence, encoder_final_f, encoder_final_b  = encoder(embedding)

encoder_final = Concatenate(axis=-1)([encoder_final_f, encoder_final_b])



decoder_inputs = Input(shape=(Max_length -1, ))
decoder_inputs_aa = Input(shape=(Max_length, ))

dex=  Embedding(input_dim = dna_vocab_size, output_dim = embedding_size_dec, trainable=True, mask_zero = True)


final_dex= dex(decoder_inputs)
final_dex_aa =  encod_emb(decoder_inputs_aa)


decoder = GRU(2*hidden_size_enc, return_sequences = True, return_state = True)
decoder_aa =  GRU(2*hidden_size_enc_aa, return_sequences = True, return_state = True)

decoder_sequence, decoder_final = decoder(final_dex, initial_state=encoder_final)
decoder_sequence_aa, decoder_final_aa = decoder_aa(final_dex_aa, initial_state=encoder_final)


attn_layer = Attention()
attn_out = attn_layer([decoder_sequence, encoder_sequence])
attn_layer_aa = Attention()
attn_out_aa = attn_layer_aa([decoder_sequence_aa, encoder_sequence])

decoder_concat_input = Concatenate(axis=-1)([decoder_sequence, attn_out]) #decoder_sequence, 
decoder_concat_input_aa = Concatenate(axis=-1)([decoder_sequence_aa, attn_out_aa]) #decoder_sequence,


Intermediate_layer = TimeDistributed(Dense(Dense_layer_size, activation='tanh'))
Intermediate_layer_aa= TimeDistributed(Dense(Dense_layer_size_aa, activation='tanh'))

Intemediate_output = Intermediate_layer(decoder_concat_input) #decoder_concat_input
Intemediate_output_aa = Intermediate_layer_aa(decoder_concat_input_aa) #decoder_concat_input


dropout_layer = Dropout(drop_rate)
dropout_output = dropout_layer(Intemediate_output)

dropout_layer_aa = Dropout(drop_rate_aa)
dropout_output_aa = dropout_layer(Intemediate_output_aa)

dense_layer = TimeDistributed(Dense(dna_vocab_size, activation='softmax'))
logits = dense_layer(dropout_output)

dense_layer_aa = TimeDistributed(Dense(aa_vocab_size, activation='softmax'))
logits_aa = dense_layer_aa(dropout_output_aa)

enc_dec_model = Model([input_sequence, decoder_inputs, decoder_inputs_aa], [logits, logits_aa])

enc_dec_model.compile(loss=sparse_categorical_crossentropy,
              optimizer=Adam(learning_rate = learning_rate),
              metrics=['accuracy'])
enc_dec_model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 1000)]       0           []                               
                                                                                                  
 embedding (Embedding)          (None, 1000, 149)    3725        ['input_1[0][0]',                
                                                                  'input_3[0][0]']                
                                                                                                  
 input_2 (InputLayer)           [(None, 999)]        0           []                               
                                                                                                  
 bidirectional (Bidirectional)  [(None, 1000, 1026)  2043792     ['embedding[0][0]']          

In [4]:
checkpoint_path = "/Desktop/PichiaData/2Target_AllData/FinArch2_AttnCorr_cp.ckpt"
checkpoint_dir = os.path.dirname(checkpoint_path)

# Create a callback that saves the model's weights
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path,
                                                 save_weights_only=True,
                                                 verbose=1)

early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", min_delta=0, patience = 3,
    verbose=0, mode="auto", baseline=None, restore_best_weights=False)
## Train the model
model_results = enc_dec_model.fit([AA_tr[:,1:Max_length+1], Cds_tr[:,0:Max_length-1], AA_tr[:,0:Max_length]], 
                                  [Cds_tr[:,1:Max_length],  AA_tr[:,1:Max_length+1]], 
                                  batch_size= batch_size, 
                                  epochs= epochs, 
                                  validation_split=0.2, callbacks=[cp_callback, early_stop])



Epoch 1/100
148/148 [==============================] - ETA: 0s - loss: 2.8780 - time_distributed_2_loss: 1.6691 - time_distributed_3_loss: 1.2090 - time_distributed_2_accuracy: 0.0500 - time_distributed_3_accuracy: 0.1136
Epoch 1: saving model to /Desktop/PichiaData/2Target_AllData\FinArch2_AttnCorr_cp.ckpt
148/148 [==============================] - 142s 895ms/step - loss: 2.8780 - time_distributed_2_loss: 1.6691 - time_distributed_3_loss: 1.2090 - time_distributed_2_accuracy: 0.0500 - time_distributed_3_accuracy: 0.1136 - val_loss: 3.0895 - val_time_distributed_2_loss: 1.7920 - val_time_distributed_3_loss: 1.2975 - val_time_distributed_2_accuracy: 0.0590 - val_time_distributed_3_accuracy: 0.1254
Epoch 2/100
148/148 [==============================] - ETA: 0s - loss: 1.8330 - time_distributed_2_loss: 1.2159 - time_distributed_3_loss: 0.6171 - time_distributed_2_accuracy: 0.2104 - time_distributed_3_accuracy: 0.5534
Epoch 2: saving model to /Desktop/PichiaData/2Target_AllData\FinArch2_At

Epoch 13/100
148/148 [==============================] - ETA: 0s - loss: 0.4258 - time_distributed_2_loss: 0.4254 - time_distributed_3_loss: 4.1002e-04 - time_distributed_2_accuracy: 0.5301 - time_distributed_3_accuracy: 0.9998
Epoch 13: saving model to /Desktop/PichiaData/2Target_AllData\FinArch2_AttnCorr_cp.ckpt
148/148 [==============================] - 133s 903ms/step - loss: 0.4258 - time_distributed_2_loss: 0.4254 - time_distributed_3_loss: 4.1002e-04 - time_distributed_2_accuracy: 0.5301 - time_distributed_3_accuracy: 0.9998 - val_loss: 0.4582 - val_time_distributed_2_loss: 0.4577 - val_time_distributed_3_loss: 5.3367e-04 - val_time_distributed_2_accuracy: 0.5411 - val_time_distributed_3_accuracy: 0.9998
Epoch 14/100
148/148 [==============================] - ETA: 0s - loss: 0.4204 - time_distributed_2_loss: 0.4199 - time_distributed_3_loss: 4.5603e-04 - time_distributed_2_accuracy: 0.5405 - time_distributed_3_accuracy: 0.9998
Epoch 14: saving model to /Desktop/PichiaData/2Target

148/148 [==============================] - 134s 903ms/step - loss: 0.2327 - time_distributed_2_loss: 0.2322 - time_distributed_3_loss: 4.5795e-04 - time_distributed_2_accuracy: 0.7979 - time_distributed_3_accuracy: 0.9998 - val_loss: 0.2011 - val_time_distributed_2_loss: 0.2005 - val_time_distributed_3_loss: 6.5145e-04 - val_time_distributed_2_accuracy: 0.8533 - val_time_distributed_3_accuracy: 0.9998
Epoch 25/100
148/148 [==============================] - ETA: 0s - loss: 0.2114 - time_distributed_2_loss: 0.2110 - time_distributed_3_loss: 4.5483e-04 - time_distributed_2_accuracy: 0.8215 - time_distributed_3_accuracy: 0.9998
Epoch 25: saving model to /Desktop/PichiaData/2Target_AllData\FinArch2_AttnCorr_cp.ckpt
148/148 [==============================] - 134s 906ms/step - loss: 0.2114 - time_distributed_2_loss: 0.2110 - time_distributed_3_loss: 4.5483e-04 - time_distributed_2_accuracy: 0.8215 - time_distributed_3_accuracy: 0.9998 - val_loss: 0.1772 - val_time_distributed_2_loss: 0.1764 -

Epoch 36/100
148/148 [==============================] - ETA: 0s - loss: 0.0974 - time_distributed_2_loss: 0.0968 - time_distributed_3_loss: 5.7330e-04 - time_distributed_2_accuracy: 0.9272 - time_distributed_3_accuracy: 0.9998
Epoch 36: saving model to /Desktop/PichiaData/2Target_AllData\FinArch2_AttnCorr_cp.ckpt
148/148 [==============================] - 133s 902ms/step - loss: 0.0974 - time_distributed_2_loss: 0.0968 - time_distributed_3_loss: 5.7330e-04 - time_distributed_2_accuracy: 0.9272 - time_distributed_3_accuracy: 0.9998 - val_loss: 0.0701 - val_time_distributed_2_loss: 0.0694 - val_time_distributed_3_loss: 6.2768e-04 - val_time_distributed_2_accuracy: 0.9624 - val_time_distributed_3_accuracy: 0.9998
Epoch 37/100
148/148 [==============================] - ETA: 0s - loss: 0.0924 - time_distributed_2_loss: 0.0919 - time_distributed_3_loss: 5.0659e-04 - time_distributed_2_accuracy: 0.9309 - time_distributed_3_accuracy: 0.9998
Epoch 37: saving model to /Desktop/PichiaData/2Target

148/148 [==============================] - 133s 902ms/step - loss: 0.0686 - time_distributed_2_loss: 0.0683 - time_distributed_3_loss: 3.5884e-04 - time_distributed_2_accuracy: 0.9466 - time_distributed_3_accuracy: 0.9998 - val_loss: 0.0476 - val_time_distributed_2_loss: 0.0470 - val_time_distributed_3_loss: 6.1624e-04 - val_time_distributed_2_accuracy: 0.9753 - val_time_distributed_3_accuracy: 0.9998
Epoch 48/100
148/148 [==============================] - ETA: 0s - loss: 0.0641 - time_distributed_2_loss: 0.0636 - time_distributed_3_loss: 4.3224e-04 - time_distributed_2_accuracy: 0.9506 - time_distributed_3_accuracy: 0.9998
Epoch 48: saving model to /Desktop/PichiaData/2Target_AllData\FinArch2_AttnCorr_cp.ckpt
148/148 [==============================] - 134s 906ms/step - loss: 0.0641 - time_distributed_2_loss: 0.0636 - time_distributed_3_loss: 4.3224e-04 - time_distributed_2_accuracy: 0.9506 - time_distributed_3_accuracy: 0.9998 - val_loss: 0.0457 - val_time_distributed_2_loss: 0.0453 -